In [1]:
import pandas as pd
import numpy as np
import os

np.random.seed(2026)
row_num = 1200
data_dict = {
    "CustomerID": np.arange(1, row_num + 1),
    "Churn": np.random.randint(0, 2, row_num),
    "Tenure": np.random.normal(loc=32, scale=14, size=row_num).round(0),
    "WarehouseToHome": np.random.normal(loc=16, scale=7, size=row_num).round(0),
    "PreferredLoginDevice": np.random.choice(["Phone", "Mobile", "Laptop"], row_num),
    "PreferredPaymentMode": np.random.choice(["COD", "CC", "Debit Card"], row_num),
    "NumberOfDeviceRegistered": np.random.randint(1, 6, row_num),
    "NumberOfAddress": np.random.randint(1, 10, row_num),
    "CashbackAmount": np.random.randint(0, 480, row_num),
    "OrderCount": np.random.randint(1, 18, row_num),
    "DaySinceLastOrder": np.random.randint(1, 55, row_num)
}
df = pd.DataFrame(data_dict)

# 手动制造缺失值，还原课程缺失值场景
miss_cols = ["Tenure", "WarehouseToHome", "CashbackAmount", "DaySinceLastOrder"]
for col in miss_cols:
    miss_idx = np.random.choice(df.index, size=80, replace=False)
    df.loc[miss_idx, col] = np.nan

print("数据集行列规模：", df.shape)
print("数据前5行预览：")
print(df.head())

# 任务1 业务问答
"""
1. 单行数据含义：单条电商用户全生命周期行为、消费、会员流失记录
2. 用户唯一标识字段：CustomerID
3. 用户流失目标字段：Churn（1=流失，0=留存）
"""

# ---------------------- 2. 缺失值报告（按缺失量降序） ----------------------
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失百分比(%)": missing_pct
}).sort_values("缺失数量", ascending=False)
print("\n===== 缺失值统计报告 =====")
print(missing_report)

# ---------------------- 3. 重复值检查 ----------------------
total_dup_rows = df.duplicated().sum()
customerid_dup = df["CustomerID"].duplicated().sum()
print(f"\n完全重复数据行数：{total_dup_rows}")
print(f"CustomerID用户ID重复记录数：{customerid_dup}")
"""
思考题：不能直接删除CustomerID重复数据
同一用户会产生多条不同时间的消费行为记录，重复ID代表用户多次下单，删除会丢失时序业务数据。
"""

# ---------------------- 4. 数值字段中位数填充 ----------------------
num_fill_cols = ["Tenure", "WarehouseToHome", "NumberOfDeviceRegistered",
                 "NumberOfAddress", "CashbackAmount", "OrderCount", "DaySinceLastOrder"]
for col in num_fill_cols:
    df[col] = df[col].fillna(df[col].median())
print("\n中位数填充后，数值字段缺失值统计：")
print(df[num_fill_cols].isna().sum())

"""
中位数填充不适用场景：
1. 数据极度偏斜，极值具备真实业务价值（大额返现、高频下单用户），中位数会抹平业务差异；
2. 缺失并非随机（特定人群才会缺失），填充后会扭曲数据分布；
替代方案：正态分布数据用均值、特征相关性高时用模型预测填充。
"""

# ---------------------- 5. 分类字段标准化统一同义文本 ----------------------
# 登录设备统一
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
    "Phone": "Mobile Phone",
    "Mobile": "Mobile Phone"
})
# 支付方式统一
df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "COD": "Cash on Delivery",
    "CC": "Credit Card"
})
print("\n标准化后登录设备分布：")
print(df["PreferredLoginDevice"].value_counts())
print("\n标准化后支付方式分布：")
print(df["PreferredPaymentMode"].value_counts())

# ---------------------- 6. IQR箱线法统计异常值检测 ----------------------
def iqr_outlier_check(data, col_name):
    q1 = data[col_name].quantile(0.25)
    q3 = data[col_name].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_cnt = data[(data[col_name] < lower_bound) | (data[col_name] > upper_bound)].shape[0]
    print(f"\n字段 {col_name} 统计候选异常值数量：{outlier_cnt}")
    print(f"统计下限：{lower_bound:.2f}  统计上限：{upper_bound:.2f}")
    return outlier_cnt

# 检测课程指定3个字段
iqr_outlier_check(df, "WarehouseToHome")
iqr_outlier_check(df, "OrderCount")
iqr_outlier_check(df, "CashbackAmount")
"""
结论：IQR仅识别统计学候选异常，不可直接删除；部分极值是真实高价值用户行为，需要结合业务判断。
"""

# ---------------------- 7. 业务逻辑异常校验 ----------------------
err_tenure = df[df["Tenure"] < 0].shape[0]
err_warehouse = df[df["WarehouseToHome"] < 0].shape[0]
err_order = df[df["OrderCount"] <= 0].shape[0]
err_cashback = df[df["CashbackAmount"] < 0].shape[0]
print(f"\n业务逻辑违规数据统计：")
print(f"使用时长负数：{err_tenure}")
print(f"仓库距离负数：{err_warehouse}")
print(f"订单数小于等于0：{err_order}")
print(f"返现金额负数：{err_cashback}")

# ---------------------- 8. 清洗验收断言校验 ----------------------
# 校验1：数值字段无缺失
assert df[num_fill_cols].isna().sum().sum() == 0, "数值字段仍存在缺失值！"
# 校验2：旧设备简写已全部清理
assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备未完成标准化"
# 校验3：旧支付简写已全部清理
assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式未完成标准化"
print("\n✅ 全部数据清洗校验通过！")

# ---------------------- 9. 导出清洗完成数据集 ----------------------
# 自动创建output文件夹
if not os.path.exists("output"):
    os.mkdir("output")
# 导出csv，utf-8-sig防止Excel中文乱码
df.to_csv("output/ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
print("\n清洗完成，文件已导出至 output/ecommerce_customer_cleaned.csv")

数据集行列规模： (1200, 11)
数据前5行预览：
   CustomerID  Churn  Tenure  WarehouseToHome PreferredLoginDevice  \
0           1      1    41.0             18.0               Mobile   
1           2      0    29.0             17.0               Mobile   
2           3      0    50.0             16.0               Laptop   
3           4      0     NaN             27.0               Laptop   
4           5      1    24.0              4.0                Phone   

  PreferredPaymentMode  NumberOfDeviceRegistered  NumberOfAddress  \
0           Debit Card                         1                3   
1                   CC                         5                2   
2                  COD                         1                5   
3                   CC                         2                6   
4                  COD                         2                1   

   CashbackAmount  OrderCount  DaySinceLastOrder  
0           115.0           5               34.0  
1           409.0          14    